# Serving, Policy y auditoría: del modelo a una evidencia verificable

Ejecuta primero el Notebook 04. Allí se decide qué **versión** es `champion`; aquí comprobamos qué ocurre cuando un proceso real la carga y atiende solicitudes. Antes del código, separa estos cuatro sustantivos:

| Concepto | Pregunta que responde | No es |
|---|---|---|
| **Registry** | ¿Qué versión registrada apunta hoy a `champion`? | Un servidor ejecutándose. |
| **Model API** | ¿Qué copia del modelo cargó este proceso al arrancar? | El Registry ni una regla de negocio. |
| **Policy** | Dada una probabilidad, ¿qué recomendación contractual corresponde? | La probabilidad ni la decisión final. |
| **Audit** | ¿Qué evidencia deja cada evaluación? | Una métrica de MLflow ni una decisión final. |

Ruta que observaremos: `champion` en Registry -> Model API -> probabilidad -> Policy -> `model_evaluations`. El Registry y el runtime se mantienen separados deliberadamente: la API carga una sola vez al iniciar.

In [1]:
import contextlib
import importlib
import io
import json
import os
import socket
import subprocess
import sys
from pathlib import Path

import httpx
import mlflow
import pandas as pd
from IPython.display import Markdown, display
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
# El override conserva solo estado técnico; SQLite operacional viene de INVOICEOPS_DB_PATH.
DEMO_ROOT = Path(
    os.environ.get("INVOICEOPS_NOTEBOOK_DEMO_ROOT", PROJECT_ROOT / "var" / "t23_5_demo")
).resolve()
STATE_PATH = DEMO_ROOT / "state.json"
if not STATE_PATH.exists():
    raise RuntimeError("Ejecuta primero el Notebook 04 con el mismo INVOICEOPS_NOTEBOOK_DEMO_ROOT.")
state = json.loads(STATE_PATH.read_text())
state.setdefault("completed_actions", {})
state.setdefault("evaluations", {})
TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")
if not TRACKING_URI:
    raise RuntimeError(
        "Preflight MLflow: falta MLFLOW_TRACKING_URI en este kernel. Inicia el servidor compartido, exporta la URI y reinicia el kernel antes de continuar."
    )
legacy_db = importlib.import_module("invoiceops.legacy.db")

DEMO_DB = legacy_db._resolve_db_path(None)
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
try:
    client.search_experiments(max_results=1)
except Exception as error:
    raise RuntimeError(
        f"Preflight MLflow: no se puede conectar a {TRACKING_URI}. Verifica que el único servidor MLflow compartido esté activo y que la URI sea accesible. Detalle: {error}"
    ) from error
display(
    Markdown(
        f"**Backend MLflow compartido preparado.** Tracking/Registry: `{TRACKING_URI}`. Estado técnico: `{STATE_PATH.name}`. Auditoría SQLite operacional: `{DEMO_DB}`."
    )
)

**Backend MLflow compartido preparado.** Tracking/Registry: `http://127.0.0.1:5000`. Estado y auditoría SQLite local: `invoiceops.db`.

## 1. Resolver el champion

**Qué estás mirando:** el alias móvil `champion` del Registry y la identidad de la versión a la que apunta ahora.

**Por qué importa:** un alias permite hablar de la versión aprobada sin escribir un número fijo. Sin embargo, saber el alias no prueba todavía qué modelo usa un proceso en memoria.

**Qué ejecuta esta celda:** consulta MLflow para resolver `models:/invoice-review@champion` y recupera su run.

**Qué deberías comprobar:** nombre, versión, run y tipo del modelo aparecen en una tabla.

**Qué significa si falla:** 04 no terminó, el alias no existe o este notebook no usa el mismo `DEMO_ROOT`. No inventes una versión: vuelve a verificar el Registry.

In [2]:
from invoiceops.ml.registry import MODEL_NAME, promote_model

champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
champion_run = client.get_run(champion.run_id)
champion_table = pd.DataFrame(
    [
        {
            "Nombre": MODEL_NAME,
            "Versión champion": f"v{champion.version}",
            "Run": champion.run_id,
            "Tipo": champion_run.data.params.get("model_type", "desconocido"),
        }
    ]
)
display(champion_table)

,Nombre,Versión champion,Run,Tipo
0,invoice-review,v2,3b82b0fa909f4fb9adb506bb64de75ed,random_forest


## Ver esta ejecución en MLflow

MLflow UI es un **visor del mismo backend compartido** que usa esta demo; no crea una copia de runs ni del Registry. El instructor ya inició ese único servidor antes de abrir Jupyter: no inicies otro servidor.

En la misma UI revisa **Experiments -> invoice-risk -> runs** para métricas, parámetros y artifacts, y **Models -> invoice-review** para versiones y aliases. Este notebook escribe `model_evaluations`, `source`, `reason` y `correlation_id` en la SQLite operacional compartida resuelta por `INVOICEOPS_DB_PATH`; se observan en las tablas del notebook o el portal, **no** en MLflow.

In [3]:
print(f"Abre la misma UI configurada en este kernel: {TRACKING_URI}")
print("Experiments -> invoice-risk -> runs; Models -> invoice-review -> versiones y aliases.")

Abre la misma UI configurada en este kernel: http://127.0.0.1:5000
Experiments -> invoice-risk -> runs; Models -> invoice-review -> versiones y aliases.


## 2. Arrancar la Model API y confirmar health

**Qué estás mirando:** un proceso local de API y la identidad del modelo que cargó al iniciar.

**Por qué importa:** Registry state no es Runtime state. Cambiar `champion` después no hace hot reload; solo un reinicio puede cargar la nueva referencia.

**Qué ejecuta esta celda:** reserva un puerto local, crea Uvicorn enlazado a `127.0.0.1`, espera `/health` y muestra la respuesta. Solo detendrá después el proceso que creó ella misma.

**Qué deberías comprobar:** `status=ok`, nombre, versión y run deben coincidir con la tabla anterior.

**Qué significa si falla:** la API no pudo cargar el alias, el entorno no tiene dependencias o el proceso no quedó sano. No continúes a `/predict`; revisa el error y el estado del Registry.

In [4]:
from notebooks._demo_helpers import cleanup_created_process, wait_for_health

api_process = None
api_created_by_notebook = False


def unused_local_port():
    with socket.socket() as listener:
        listener.bind(("127.0.0.1", 0))
        return listener.getsockname()[1]


def start_api():
    global api_process, api_created_by_notebook, BASE_URL
    cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
    port = unused_local_port()
    BASE_URL = f"http://127.0.0.1:{port}"
    environment = os.environ | {
        "MLFLOW_TRACKING_URI": TRACKING_URI,
        "INVOICEOPS_MODEL_URI": f"models:/{MODEL_NAME}@champion",
        "PYTHONPATH": str(PROJECT_ROOT / "src"),
    }
    venv_python = (
        PROJECT_ROOT / ".venv" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
    )
    python_executable = str(venv_python) if venv_python.is_file() else sys.executable
    api_process = subprocess.Popen(
        [
            python_executable,
            "-m",
            "uvicorn",
            "invoiceops.model_api.app:app",
            "--host",
            "127.0.0.1",
            "--port",
            str(port),
        ],
        cwd=PROJECT_ROOT,
        env=environment,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )
    api_created_by_notebook = True

    def health_request(url):
        try:
            return httpx.get(url, timeout=1).status_code == 200
        except httpx.RequestError:
            return False

    try:
        wait_for_health(
            f"{BASE_URL}/health",
            timeout_seconds=20,
            poll_interval_seconds=0.25,
            request=health_request,
        )
    except Exception:
        cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
        raise


start_api()
health = httpx.get(f"{BASE_URL}/health", timeout=5).json()
display(
    pd.DataFrame(
        [
            {
                "Estado": health["status"],
                "Nombre": health["model_name"],
                "Versión cargada": f"v{health['model_version']}",
                "Run cargado": health["run_id"],
            }
        ]
    )
)

,Estado,Nombre,Versión cargada,Run cargado
0,ok,invoice-review,v2,3b82b0fa909f4fb9adb506bb64de75ed


## 3. Features, `/predict` y tres mecanismos distintos

**Qué estás mirando:** para `INV-10029` e `INV-10030`, la recomendación de Rule v1, la probabilidad devuelta por el modelo y la recomendación final de Policy.

**Por qué importa:** Rule v1 usa reglas explícitas; el modelo estima una probabilidad con ocho features; Policy transforma esa probabilidad mediante su umbral contractual. Son mecanismos diferentes aunque puedan coincidir. Los resultados son datos reales de esta ejecución: ninguna factura tiene una decisión ML prefijada.

**Qué ejecuta esta celda:** inicializa la SQLite operacional compartida resuelta por `INVOICEOPS_DB_PATH`, transforma cada factura con `invoice_to_features`, llama `/predict` y aplica `recommend_from_probability`. El estado técnico auxiliar permanece separado.

**Qué deberías comprobar:** compara cada fila, incluyendo versión/run del modelo, Rule v1, probabilidad y Policy. La igualdad o diferencia entre recomendaciones se interpreta a partir de la salida real.

**Qué significa si falla:** un contrato de features o la API está fallando; no sustituyas la respuesta por una probabilidad inventada.

In [5]:
from invoiceops.domain.policy import fallback_recommendation, recommend_from_probability
from invoiceops.domain.rules import decide_invoice
from invoiceops.legacy.db import (
    get_invoice,
    init_db,
    insert_model_evaluation,
    list_model_evaluations,
)
from invoiceops.legacy.seed import seed_invoices
from invoiceops.ml.features import invoice_to_features

with contextlib.redirect_stdout(io.StringIO()):
    init_db(DEMO_DB)
if get_invoice(DEMO_DB, "INV-10029") is None:
    seed_invoices(DEMO_DB)


def predict_and_recommend(invoice_id):
    invoice = get_invoice(DEMO_DB, invoice_id)
    response = httpx.post(f"{BASE_URL}/predict", json=invoice_to_features(invoice), timeout=5)
    response.raise_for_status()
    prediction = response.json()
    return invoice, prediction, recommend_from_probability(prediction["manual_review_probability"])


comparison_rows = []
for invoice_id in ("INV-10029", "INV-10030"):
    invoice, prediction, recommendation = predict_and_recommend(invoice_id)
    comparison_rows.append(
        {
            "Factura": invoice_id,
            "Rule v1": decide_invoice(invoice).value,
            "Probabilidad": prediction["manual_review_probability"],
            "Policy": recommendation.decision.value,
            "Versión Policy": recommendation.policy_version,
            "Versión modelo": f"v{prediction['model_version']}",
            "Run": prediction["run_id"],
        }
    )
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.style.format({"Probabilidad": "{:.3f}"}))
print("Comparación completada con respuestas reales de /predict; no hay decisiones ML prefijadas.")

0 migrations pending


,Factura,Rule v1,Probabilidad,Policy,Versión Policy,Versión modelo,Run
0,INV-10029,AUTO_PROCESS,0.240,AUTO_PROCESS,ml-policy-v1,v2,3b82b0fa909f4fb9adb506bb64de75ed
1,INV-10030,AUTO_PROCESS,0.770,AUTO_PROCESS,ml-policy-v1,v2,3b82b0fa909f4fb9adb506bb64de75ed


Comparación completada con respuestas reales de /predict; no hay decisiones ML prefijadas.


## 4. Promotion/restart A/B y evidencia persistida

**Qué estás mirando:** el ciclo completo para cada candidato: mover alias, detener solo la API creada aquí, iniciar API, confirmar runtime en `/health`, predecir, persistir y comparar.

**Por qué importa:** una Promotion cambia Registry, no la memoria del servidor. El restart y `/health` son la evidencia de que el runtime realmente cambió.

**Qué ejecuta esta celda:** **⚠ MODIFICA ESTADO** en el Registry y SQLite operacional. Sus acciones son idempotentes: una segunda ejecución no duplica promociones ni auditorías.

**Qué deberías comprobar:** A y B muestran versión, run, probabilidad y Policy; la tabla de auditoría añade `source`, `reason` y `correlation_id`. Que ambos resultados coincidan no elimina el valor de la trazabilidad: son versiones distintas.

**Qué significa si falla:** si `/health` no confirma la versión promovida, hay una discrepancia Registry/Runtime y no se debe persistir como si el cambio hubiera ocurrido.

In [6]:
# ⚠ MODIFICA ESTADO: Promotion y auditoría operacional, protegidas por identidad durable.
from notebooks._demo_helpers import run_mutable_action_once


def save_state():
    STATE_PATH.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n")


def promote_for_audit(candidate):
    version = str(state["registered_versions"][candidate])

    def promote_once():
        current = str(client.get_model_version_by_alias(MODEL_NAME, "champion").version)
        if current != version:
            promote_model(version)
        return version

    promoted = run_mutable_action_once(
        f"promote-for-audit-{candidate}", state["completed_actions"], promote_once
    )
    save_state()
    return str(promoted)


def serve_and_persist(candidate):
    version = promote_for_audit(candidate)

    def persist_once():
        start_api()  # El runtime cambia solo después del reinicio.
        runtime = httpx.get(f"{BASE_URL}/health", timeout=5).json()
        assert runtime["model_version"] == version
        invoice, prediction, recommendation = predict_and_recommend("INV-10030")
        correlation_id = (
            f"notebook-05:{invoice.invoice_id}:model:{candidate}:v{prediction['model_version']}"
        )
        insert_model_evaluation(
            DEMO_DB,
            invoice.invoice_id,
            correlation_id=correlation_id,
            recommendation=recommendation,
            model_name=prediction["model_name"],
            model_version=prediction["model_version"],
            run_id=prediction["run_id"],
            manual_review_probability=prediction["manual_review_probability"],
        )
        return {
            "Candidato": candidate,
            "Versión": f"v{prediction['model_version']}",
            "Run": prediction["run_id"],
            "Probabilidad": prediction["manual_review_probability"],
            "Policy": recommendation.decision.value,
        }

    state["evaluations"][candidate] = run_mutable_action_once(
        f"persist-INV-10030-{candidate}", state["completed_actions"], persist_once
    )
    save_state()


for candidate in ("A", "B"):
    serve_and_persist(candidate)
display(
    pd.DataFrame([state["evaluations"]["A"], state["evaluations"]["B"]]).style.format(
        {"Probabilidad": "{:.3f}"}
    )
)
audit = pd.DataFrame([dict(row) for row in list_model_evaluations(DEMO_DB, "INV-10030")])
display(
    audit[
        [
            "invoice_id",
            "model_name",
            "model_version",
            "run_id",
            "manual_review_probability",
            "recommendation",
            "source",
            "reason",
            "correlation_id",
        ]
    ]
    .rename(
        columns={
            "invoice_id": "Factura",
            "model_name": "Nombre",
            "model_version": "Versión",
            "run_id": "Run",
            "manual_review_probability": "Probabilidad",
            "recommendation": "Policy",
            "correlation_id": "Correlation ID",
        }
    )
    .style.format({"Probabilidad": "{:.3f}"})
)

,Candidato,Versión,Run,Probabilidad,Policy
0,A,v1,91214619c9b94267a2308374379b74a6,0.770,AUTO_PROCESS
1,B,v2,3b82b0fa909f4fb9adb506bb64de75ed,0.770,AUTO_PROCESS


,Factura,Nombre,Versión,Run,Probabilidad,Policy,source,reason,Correlation ID
0,INV-10030,invoice-review,2,3b82b0fa909f4fb9adb506bb64de75ed,0.770,AUTO_PROCESS,model,probability_below_threshold,t23-5-b-54e61c0f-fe90-4ec3-8cda-76c771a2673f
1,INV-10030,invoice-review,1,91214619c9b94267a2308374379b74a6,0.770,AUTO_PROCESS,model,probability_below_threshold,t23-5-a-bff7e8e6-83b6-48c5-9deb-9a910817f41b


## 5. Fallback: control de riesgo, no juicio sobre el modelo

**Qué estás mirando:** la respuesta segura cuando el modelo no está disponible.

**Por qué importa:** fallback no significa que el modelo sea malo. Significa que el servicio o su dependencia no está disponible y el sistema elige controlar el riesgo antes que inventar evidencia.

**Qué ejecuta esta celda:** detiene solo el proceso creado por este notebook y persiste una recomendación de fallback. **⚠ MODIFICA ESTADO** únicamente en la auditoría operacional; es idempotente.

**Qué deberías comprobar:** `MANUAL_REVIEW`, `source=fallback`, `reason=model_unavailable` y probabilidad `null`.

**Qué significa si falla:** la limpieza o la persistencia de evidencia falló; no reemplaces `null` por `1.0`, porque nunca se obtuvo un score.

In [7]:
# ⚠ MODIFICA ESTADO: registra fallback sin fabricar una probabilidad.
cleanup_created_process(api_process, created_by_helper=api_created_by_notebook)
api_process = None
api_created_by_notebook = False
fallback = fallback_recommendation()


def persist_fallback():
    correlation_id = "notebook-05:INV-10030:fallback"
    insert_model_evaluation(
        DEMO_DB,
        "INV-10030",
        correlation_id=correlation_id,
        recommendation=fallback,
        model_name=None,
        model_version=None,
        run_id=None,
        manual_review_probability=None,
    )
    return {
        "Factura": "INV-10030",
        "Policy": fallback.decision.value,
        "source": fallback.source,
        "reason": fallback.reason,
        "Probabilidad": None,
        "Correlation ID": correlation_id,
    }


fallback_result = run_mutable_action_once(
    "persist-fallback-INV-10030", state["completed_actions"], persist_fallback
)
state["evaluations"]["fallback"] = fallback_result
save_state()
display(pd.DataFrame([fallback_result]))

,Factura,Policy,source,reason,Probabilidad,Correlation ID
0,INV-10030,MANUAL_REVIEW,fallback,model_unavailable,None,t23-5-fallback-b553847c-ffc4-48a0-8998-128e806...


## Idea de cierre

Una versión `champion` define una referencia en Registry. Un reinicio más `/health` demuestra qué versión usa el runtime. `/predict` aporta una probabilidad; `ml-policy-v1` entrega una recomendación; la auditoría deja contexto para reconstruir la evaluación. La decisión final sigue siendo una responsabilidad separada.